## Function to build Silver matches

In [0]:
from pyspark.sql.functions import col, lit, to_date, year, expr

def build_silver_matches(bronze_path: str, silver_path: str, match_type: str):
    """
    Reads ATP match CSVs from Bronze, normalizes ranks and IDs, and writes to Silver Delta.
    Supports singles, doubles, amateur, futures, challengers.
    """
    # Read Bronze CSVs
    df = spark.read.option("header", True).option("inferSchema", True).csv(bronze_path)

    # Handle ranks / IDs based on match type
    if match_type == "doubles":
        # Average ranks for doubles
        df = df.withColumn(
            "winner_rank",
            expr("(coalesce(winner1_rank,0) + coalesce(winner2_rank,0))/2")
        ).withColumn(
            "loser_rank",
            expr("(coalesce(loser1_rank,0) + coalesce(loser2_rank,0))/2")
        )
        # Representative winner/loser ID
        df = df.withColumn("winner_id", col("winner1_id")).withColumn("loser_id", col("loser1_id"))

    else:
        # Singles, amateur, futures, challengers
        for col_name in ["winner_rank", "loser_rank", "winner_id", "loser_id"]:
            if col_name not in df.columns:
                df = df.withColumn(col_name, lit(None).cast("int"))

    # Common transformations
    silver_df = (
        df
        .withColumn("tourney_date", to_date(col("tourney_date"), "yyyyMMdd"))
        .withColumn("match_year", year(col("tourney_date")))
        .withColumn("best_of", col("best_of").cast("int") if "best_of" in df.columns else lit(None))
        .withColumn("match_type", lit(match_type))
        .select(
            "tourney_id",
            "tourney_name",
            "surface",
            "draw_size",
            "tourney_level",
            "tourney_date",
            "match_year",
            "round",
            "winner_id",
            "loser_id",
            "winner_rank",
            "loser_rank",
            "score",
            "best_of",
            "match_type"
        )
    )

    # Write Delta partitioned by match_year
    silver_df.write.format("delta").mode("overwrite").partitionBy("match_year").save(silver_path)

## Build all Silver match tables

In [0]:
# Singles
build_silver_matches(
    bronze_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/bronze/matches/singles/*.csv",
    silver_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/silver/matches/singles",
    match_type="singles"
)

In [0]:
# Doubles
build_silver_matches(
    bronze_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/bronze/matches/doubles/*.csv",
    silver_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/silver/matches/doubles",
    match_type="doubles"
)

In [0]:
# Challenger / Qualifiers
build_silver_matches(
    bronze_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/bronze/matches/challengers_qual/*.csv",
    silver_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/silver/matches/challengers_qual",
    match_type="challengers_qual"
)


In [0]:
# Futures
build_silver_matches(
    bronze_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/bronze/matches/futures/*.csv",
    silver_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/silver/matches/futures",
    match_type="futures"
)

In [0]:
# Amateur
build_silver_matches(
    bronze_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/bronze/matches/amateur/*.csv",
    silver_path="abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/silver/matches/amateur",
    match_type="amateur"
)

## Build  players table

In [0]:
df_players = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv("abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/bronze/players/*.csv")
)

df_players_clean = (
    df_players
    .withColumn("dob", to_date(col("dob"), "yyyy-MM-dd"))
    .withColumn("height", col("height").cast("int"))
)

df_players_clean.write.format("delta").mode("overwrite").save(
    "abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/silver/players"
)

## Build rankings table

In [0]:
df_rankings = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv("abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/bronze/rankings/*.csv")
)

df_rankings_clean = (
    df_rankings
    .withColumn("ranking_date", to_date(col("ranking_date"), "yyyyMMdd"))
    .withColumn("rank", col("rank").cast("int"))
    .withColumn("points", col("points").cast("int"))
    .withColumn("ranking_year", year(col("ranking_date")))
)

df_rankings_clean.write.format("delta").mode("overwrite").partitionBy("ranking_year").save(
    "abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/silver/rankings"
)

## Data quality checks

### Loading a dataframe from "/silver/matches/singles/"

In [0]:

df = spark.read.format("delta").load("abfss://lake@stadaiccdapadevh96ndt.dfs.core.windows.net/DEV/wbielski/archive/de_project_etl_atp_tennis_matches/silver/matches/singles/")

df.display(5)

tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_year,round,winner_id,loser_id,winner_rank,loser_rank,score,best_of,match_type
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,112411,110196,null,null,6-1 7-5,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,126914,209536,null,null,6-1 6-1,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,209523,209535,null,null,6-2 6-2,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,100084,209534,null,null,6-1 6-1,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,100132,209533,null,null,6-2 6-4,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,207073,209532,null,null,4-6 8-6 6-2,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,109783,125672,null,null,6-0 9-7,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,109745,125716,null,null,6-1 6-1,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,201749,113401,null,null,6-1 6-0,3,singles
1968-2029,Dublin,Grass,32,A,1968-07-08,1968,R32,209525,110045,null,null,4-6 6-4 6-0,3,singles


### Completeness checks
Check mandatory columns for nulls

In [0]:
from pyspark.sql.functions import col

mandatory_cols = [
    "tourney_id", "tourney_name", "tourney_date", "round",
    "winner_id", "loser_id", "score"
]

null_summary = {}

for c in mandatory_cols:
    if c not in df.columns:
        null_summary[c] = "MISSING_COLUMN"
        print(f"Column {c} is missing from DataFrame")
    else:
        null_count = df.filter(col(c).isNull()).count()
        null_summary[c] = null_count
        print(f"Null values in {c}: {null_count}")

Null values in tourney_id: 0
Null values in tourney_name: 0
Null values in tourney_date: 0
Null values in round: 0
Null values in winner_id: 0
Null values in loser_id: 0
Null values in score: 3


### Uniqueness checks
Match uniqueness: no duplicate matches for the same tourney/round/winner/loser/date

In [0]:
duplicate_count = (
    df.groupBy("tourney_id", "round", "winner_id", "loser_id", "tourney_date")
      .count()
      .filter(col("count") > 1)
      .count()
)

print(f"Duplicate matches found: {duplicate_count}")

Duplicate matches found: 13


### Referential integrity / logical checks

- Winner and loser IDs should not be the same
- Ranks should be positive integers
- best_of should be 3 or 5 (ATP matches)

In [0]:
# Winner != Loser
bad_winner_loser = df.filter(col("winner_id") == col("loser_id")).count()
print(f"Matches where winner=loser: {bad_winner_loser}")

# Winner/loser ranks positive
bad_ranks = df.filter((col("winner_rank") < 1) | (col("loser_rank") < 1)).count()
print(f"Matches with invalid ranks: {bad_ranks}")

# Best-of is 3 or 5
invalid_best_of = df.filter(~col("best_of").isin([3, 5])).count()
print(f"Matches with invalid best_of: {invalid_best_of}")

Matches where winner=loser: 3
Matches with invalid ranks: 0
Matches with invalid best_of: 36


### Date & year checks

- tourney_date should not be null and match match_year
- No future dates

In [0]:
from pyspark.sql.functions import year, current_date

# match_year consistency
year_mismatch = df.filter(year(col("tourney_date")) != col("match_year")).count()
print(f"Matches with inconsistent year: {year_mismatch}")

# No future dates
future_dates = df.filter(col("tourney_date") > current_date()).count()
print(f"Matches with future tourney_date: {future_dates}")

Matches with inconsistent year: 0
Matches with future tourney_date: 0


### Distribution & outliers

- Check number of matches per year (sudden spikes could indicate duplicates)
- Check number of matches per player

In [0]:
df.groupBy("match_year").count().orderBy("match_year").show(df.count(), False)
df.groupBy("winner_id").count().orderBy(col("count").desc()).show(10)

+----------+-----+
|match_year|count|
+----------+-----+
|1967      |14   |
|1968      |4433 |
|1969      |3095 |
|1970      |3287 |
|1971      |3720 |
|1972      |3577 |
|1973      |4397 |
|1974      |4153 |
|1975      |4195 |
|1976      |3885 |
|1977      |4125 |
|1978      |3852 |
|1979      |3990 |
|1980      |3982 |
|1981      |3910 |
|1982      |4074 |
|1983      |3489 |
|1984      |3248 |
|1985      |3388 |
|1986      |3295 |
|1987      |3577 |
|1988      |3671 |
|1989      |3583 |
|1990      |3743 |
|1991      |3727 |
|1992      |3730 |
|1993      |3890 |
|1994      |3938 |
|1995      |3800 |
|1996      |3836 |
|1997      |3561 |
|1998      |3591 |
|1999      |3334 |
|2000      |3378 |
|2001      |3400 |
|2002      |3213 |
|2003      |3125 |
|2004      |3288 |
|2005      |3264 |
|2006      |3267 |
|2007      |3285 |
|2008      |3030 |
|2009      |3085 |
|2010      |3030 |
|2011      |3015 |
|2012      |3094 |
|2013      |2944 |
|2014      |2816 |
|2015      |2943 |
|2016      |

### Automating QA logs

In [0]:
dq_results = {
    "nulls": {},
    "duplicates": duplicate_count,
    "winner_loser_same": bad_winner_loser,
    "invalid_ranks": bad_ranks,
    "invalid_best_of": invalid_best_of,
    "year_mismatch": year_mismatch,
    "future_dates": future_dates
}

for k, v in dq_results.items():
    print(f"{k}: {v}")

nulls: {}
duplicates: 13
winner_loser_same: 3
invalid_ranks: 0
invalid_best_of: 36
year_mismatch: 0
future_dates: 0
